# 06 — Foundation model

Chronos-T5 applied zero-shot. **Univariate**: the model sees only the target
history, no calendar features, no weather, no sensors.

**Report sections fed:** 8 (Foundation model).

Requires `pip install torch chronos-forecasting`. CPU is sufficient.


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, stationarity
from appliance_energy.models import benchmarks, feature_models, foundation, sarimax

pd.set_option("display.width", 140)


In [ ]:
frame = data.load_hourly()
y = frame[config.TARGET]

y_train, y_test = data.train_test_split(y)
test_index = y_test.index

print(f"train {y_train.index.min()} -> {y_train.index.max()}  ({len(y_train)})")
print(f"test  {test_index.min()} -> {test_index.max()}  ({len(y_test)})")


In [ ]:
print("available:", foundation.is_available())


## Zero-shot rolling-origin forecast

512 hours of context per origin, 20 sample paths, pointwise median as the point
forecast — the appropriate summary under absolute error loss.


In [ ]:
import time

t0 = time.time()
chronos = foundation.chronos_forecaster(device="cpu")
pred = benchmarks.rolling_origin_forecast(y, test_index, config.HORIZON, chronos)
print(f"{time.time() - t0:.1f}s for {config.N_ORIGINS} origins")

evaluation.evaluate_all({"chronos_zeroshot": pred}, y_test, y_train).round(3)


## Sampling variance

With 20 samples the median carries non-trivial Monte Carlo error. Repeat runs
show how much of any difference from the other models is noise.


In [ ]:
runs = []
for seed in range(3):
    np.random.seed(seed)
    f = foundation.chronos_forecaster(device="cpu")
    p = benchmarks.rolling_origin_forecast(y, test_index, config.HORIZON, f)
    runs.append(evaluation.mase(y_test, p, y_train))

print([round(m, 4) for m in runs])
print(f"spread {max(runs) - min(runs):.4f}")


### Prediction intervals\n\nQuantiles recorded per origin during the run above.

In [ ]:
quantiles = pd.concat(chronos.quantiles_, ignore_index=True)
quantiles.index = test_index

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_index[:72], y_test.iloc[:72], color="black", lw=2, label="actual")
ax.plot(test_index[:72], pred.iloc[:72], color="tab:red", lw=1.3, label="median")
ax.fill_between(test_index[:72], quantiles["q0.1"].iloc[:72],
                quantiles["q0.9"].iloc[:72], alpha=0.2, color="tab:red",
                label="10-90% interval")
ax.legend(frameon=False)
ax.set_title("Chronos zero-shot, first 72 test hours")
fig.tight_layout()


### Empirical interval coverage\n\nNominal is 80%.

In [ ]:
inside = ((y_test >= quantiles["q0.1"]) & (y_test <= quantiles["q0.9"])).mean()
print(f"empirical coverage of the nominal 80% interval: {inside:.1%}")
